# Task 3 — Event Impact Modeling

This notebook builds the **event–indicator association matrix** from the unified dataset’s `impact_link` records, joins them to event metadata, and exports: 
- `models/event_indicator_matrix.csv`
- `reports/figures/event_indicator_matrix.png`

**Interpretation:** signed impact = `impact_magnitude × direction(+/-)`; lag is kept as metadata for later modeling.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

from src.data import load_enriched_unified, load_raw_impact_links
from src.impact_model import build_event_indicator_matrix

PROJECT_DIR = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
df = load_enriched_unified(PROJECT_DIR)
impact_links = load_raw_impact_links(PROJECT_DIR)
res = build_event_indicator_matrix(df, impact_links)
res.long_table.head(10)

In [ ]:
# Export tables
out_models = PROJECT_DIR / 'models'
out_models.mkdir(parents=True, exist_ok=True)

res.matrix.to_csv(out_models / 'event_indicator_matrix.csv')
res.long_table.to_csv(out_models / 'event_indicator_links_long.csv', index=False)

print('Wrote:', out_models / 'event_indicator_matrix.csv')
print('Wrote:', out_models / 'event_indicator_links_long.csv')

In [ ]:
# Heatmap (and save)
figures_dir = PROJECT_DIR / 'reports' / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)

m = res.matrix.fillna(0)
idx = m.index.to_frame(index=False)
y = idx['event_id'].astype(str).tolist() if 'event_id' in idx.columns else list(range(len(m)))

fig = go.Figure(
    data=go.Heatmap(
        z=m.to_numpy(),
        x=m.columns.astype(str).tolist(),
        y=y,
        colorscale='RdBu',
        zmid=0,
        colorbar=dict(title='Signed impact'),
    )
)
fig.update_layout(title='Event–Indicator Association Matrix', xaxis_title='Indicator', yaxis_title='Event ID')
fig

# Requires kaleido (in requirements.txt)
out_png = figures_dir / 'event_indicator_matrix.png'
fig.write_image(str(out_png), scale=2)
print('Saved:', out_png)